[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/19.%20The%20SLAP%20for%20Reefer%20Containers%20Problem/P19-Tier-6_executed.ipynb)

---


# 19. The SLAP for Reefer Containers Problem
## Tier 6: Autonomous Self-Optimizing Ecosystem (Distributed Intelligence)

### Purpose and scope

This notebook demonstrates a market-based, multi-agent simulation for assigning reefer containers to storage locations. Container agents submit bids for locations; location agents submit asks; an auctioneer coordinates the matching and records contracts. The implementation retains this distributed-negotiation approach while simulating its agents and auctioneer in one Python process.

The example is intended to make the negotiation and state-update logic visible. It is not a validated terminal-wide optimization model, a production Contract Net Protocol implementation, or evidence that the resulting assignments are globally optimal. In particular, the code does not implement agent-to-agent network communication, a learning algorithm, strategic coalition formation, or an empirical comparison with centralized optimization.

### Inputs and outputs

The inputs are eight containers, four locations, an assignment-cost table, and agent attributes such as power demand, temperature requirement, preferences, urgency, flexibility, and budget. The simulation returns auction bids, accepted contracts, agent and location states, and descriptive plots of the recorded rounds.

### Model assumptions

- A container can receive at most one assignment. A location can receive multiple containers over successive rounds, subject to its power capacity.
- A bid names the one feasible location the container agent currently prefers most. An ask names one location. Only bids and asks for the same location can form a contract.
- Temperature compatibility is checked before a bid is generated. Power capacity is checked both when generating a bid and when a contract is formed.
- The location's ask is an asking price, not an optimization variable. A contract is formed only when the container's bid is at least the location's ask; the contract price is their arithmetic mean.
- The auctioneer is a centralized coordinator in this simulation. Consequently, this is a market-based allocation demonstration, not a fully decentralized implementation.
- Temperatures, energy demands, costs, and prices use the supplied example's numeric scales. The example omits handling costs, travel distances, reefer plug layout, electrical-network constraints, time-varying temperatures, and uncertainty.

### Agents, negotiation, and matching

An agent is an entity that holds state and follows rules to make or respond to decisions. A container agent evaluates locations and submits a bid; a location agent calculates an ask; the auctioneer coordinates the negotiation. The Contract Net Protocol is a general framework in which a coordinator announces a task, receives proposals, awards contracts, and records commitments. The code uses this announcement–bid/ask–award pattern as a simplified simulation; it does not implement the full protocol's message exchange or failure handling.

For container \(i\) and location \(j\), let \(e_i\) be the container's power demand, \(r_i\) its required temperature, and \([T_j^{\min},T_j^{\max}]\) the location's temperature range. A location is feasible when

$$
F_{ij}=\mathbf{1}\left\{e_i\leq P_j^{\mathrm{cap}}-P_j^{\mathrm{used}},\quad T_j^{\min}\leq r_i\leq T_j^{\max}\right\}=1.
$$

Here, \(P_j^{\mathrm{cap}}\) is the location's power capacity, \(P_j^{\mathrm{used}}\) is its current power use, and the indicator \(\mathbf{1}\{\cdot\}\) equals one when both conditions hold. The code checks these conditions in `_generate_bid` and checks capacity again before creating a contract.

Among feasible locations, the container selects the location with the greatest weighted utility:

$$
U_{ij}=0.4\,u_j+0.3\,t_{ij}+0.3\,p_{ij},
$$

where \(u_j=(P_j^{\mathrm{cap}}-P_j^{\mathrm{used}})/P_j^{\mathrm{cap}}\) is the remaining-capacity fraction, \(t_{ij}=1\) for a temperature-compatible location and \(0\) otherwise, and \(p_{ij}\) is the container's stated preference for the location type. Since only feasible locations are considered, \(t_{ij}=1\) for every candidate evaluated by the code. The weights are the illustrative values used in this example; utility is a ranking score, not a monetary quantity.

For the selected location, the bid is computed as

$$
B_i=c_{ij}\left(\frac{e_i}{50}\right)
\left(1+\frac{50q_i}{100}\right)
\left(1+\frac{d_i}{120}\right)
\left[1-0.2(1-f_i)\right],
$$

where \(c_{ij}\) is the assignment cost, \(q_i\) is urgency, \(d_i\) is dwell time, and \(f_i\) is flexibility. A location with base cost \(a_j\), reputation \(r_j^{\mathrm{rep}}\), capacity utilization \(v_j\), and demand factor \(m_j\) asks

$$
A_j=a_jm_jr_j^{\mathrm{rep}}\,2(1+v_j).
$$

The price factors are illustrative rules, not estimates of real market value. The code accepts a feasible bid–ask pair when \(B_i\geq A_j\) and sets the contract price to \(C_{ij}=(B_i+A_j)/2\). It repeatedly considers feasible pairs, prioritizing the greatest bid, and removes each awarded container and location offer from that round. Because each location submits one ask per round, it can receive at most one new contract in that round; unused capacity can be offered in a later round.

This greedy negotiation rule is not an assignment optimization model: it does not minimize total assignment cost or guarantee a globally optimal allocation. The market-clearing procedure and its order are the solution method demonstrated here.

### Learning and interpretation

The agents store fields for satisfaction, reputation, experience, and learning rates to illustrate state that a richer agent model could use. This notebook does not update bidding strategies through experience or demonstrate convergence to a market equilibrium. Satisfaction is updated when a contract is awarded, while the recorded prices and performance measures are descriptive simulation outputs. They should not be interpreted as proof of learning, optimality, scalability, or superiority over another method.

### What to examine

- Which containers submit bids, and which bids meet a location's ask?
- How do temperature compatibility, remaining power capacity, preferences, and budgets affect feasible offers?
- How do the accepted contracts change container and location state?
- What do the recorded prices and utilization values describe, and what conclusions do they not establish?

In [ ]:
import math
import time
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

print("Packages imported successfully for the autonomous ecosystem simulation.")

In [ ]:
@dataclass
class ContainerAgent:
    """Agent representing one reefer container and its assignment preferences."""

    id: int
    energy: float
    temp_req: float
    temp_tolerance: float
    dwell_time: float
    cargo_type: str
    cargo_value: float
    urgency: float
    flexibility: float
    budget: float
    preferences: Dict[str, float]

    def __post_init__(self) -> None:
        self.current_location: Optional[int] = None
        self.assigned = False
        self.utility = 0.0
        self.satisfaction = 0.5
        self.learning_rate = 0.1
        self.experience: List[Dict[str, Any]] = []
        self.reputation = 0.5


@dataclass
class LocationAgent:
    """Agent representing a storage location with a finite power capacity."""

    id: int
    power_cap: float
    temp_min: float
    temp_max: float
    base_cost: float
    backup_power: bool
    location_type: str

    def __post_init__(self) -> None:
        self.assigned_containers: List[int] = []
        self.power_usage = 0.0
        self.utilization = 0.0
        self.revenue = 0.0
        self.market_price = self.base_cost
        self.demand_history = deque(maxlen=100)
        self.learning_rate = 0.05
        self.reputation = 0.5


@dataclass
class Auctioneer:
    """Coordinator that records auctions and market statistics."""

    def __post_init__(self) -> None:
        self.current_auction: Optional[Dict[str, Any]] = None
        self.auction_history: List[Dict[str, Any]] = []
        self.market_statistics: Dict[str, List[float]] = defaultdict(list)


@dataclass
class Contract:
    """Agreement recording a container, location, price, and duration."""

    container_id: int
    location_id: int
    price: float
    duration: float

    def __post_init__(self) -> None:
        self.satisfaction_score = 0.5
        self.termination_clause = False


@dataclass
class Message:
    """Timestamped message record used by the simulation's message bus."""

    sender: str
    receiver: str
    message_type: str
    content: Any

    def __post_init__(self) -> None:
        self.timestamp = time.time()
        self.priority = 1.0

print("Agent and coordination classes are defined.")

In [ ]:
containers = [
    ContainerAgent(
        id=1, energy=45, temp_req=-18, temp_tolerance=2.0, dwell_time=96,
        cargo_type="Pharmaceutical", cargo_value=500.0, urgency=0.9,
        flexibility=0.3, budget=100.0,
        preferences={"Ultra-Cold": 0.9, "Cold": 0.6, "Cool": 0.2, "General": 0.1},
    ),
    ContainerAgent(
        id=2, energy=65, temp_req=3, temp_tolerance=1.5, dwell_time=72,
        cargo_type="Fresh Produce", cargo_value=80.0, urgency=0.7,
        flexibility=0.6, budget=60.0,
        preferences={"Ultra-Cold": 0.3, "Cold": 0.8, "Cool": 0.7, "General": 0.4},
    ),
    ContainerAgent(
        id=3, energy=50, temp_req=-20, temp_tolerance=1.0, dwell_time=120,
        cargo_type="Frozen Seafood", cargo_value=120.0, urgency=0.8,
        flexibility=0.2, budget=150.0,
        preferences={"Ultra-Cold": 0.9, "Cold": 0.7, "Cool": 0.3, "General": 0.1},
    ),
    ContainerAgent(
        id=4, energy=35, temp_req=5, temp_tolerance=2.5, dwell_time=48,
        cargo_type="Dairy Products", cargo_value=60.0, urgency=0.4,
        flexibility=0.8, budget=40.0,
        preferences={"Ultra-Cold": 0.2, "Cold": 0.6, "Cool": 0.8, "General": 0.7},
    ),
    ContainerAgent(
        id=5, energy=55, temp_req=-15, temp_tolerance=1.8, dwell_time=84,
        cargo_type="Chemicals", cargo_value=200.0, urgency=0.9,
        flexibility=0.4, budget=180.0,
        preferences={"Ultra-Cold": 0.8, "Cold": 0.7, "Cool": 0.4, "General": 0.2},
    ),
    ContainerAgent(
        id=6, energy=40, temp_req=2, temp_tolerance=2.0, dwell_time=60,
        cargo_type="Beverages", cargo_value=40.0, urgency=0.3,
        flexibility=0.7, budget=30.0,
        preferences={"Ultra-Cold": 0.1, "Cold": 0.4, "Cool": 0.8, "General": 0.9},
    ),
    ContainerAgent(
        id=7, energy=60, temp_req=-22, temp_tolerance=0.8, dwell_time=108,
        cargo_type="Vaccines", cargo_value=800.0, urgency=1.0,
        flexibility=0.1, budget=300.0,
        preferences={"Ultra-Cold": 1.0, "Cold": 0.8, "Cool": 0.3, "General": 0.1},
    ),
    ContainerAgent(
        id=8, energy=48, temp_req=-12, temp_tolerance=2.2, dwell_time=96,
        cargo_type="Ice Cream", cargo_value=90.0, urgency=0.5,
        flexibility=0.6, budget=70.0,
        preferences={"Ultra-Cold": 0.7, "Cold": 0.8, "Cool": 0.6, "General": 0.3},
    ),
]

locations = [
    LocationAgent(
        id=1, power_cap=120, temp_min=-30, temp_max=-5, base_cost=15,
        backup_power=True, location_type="Ultra-Cold",
    ),
    LocationAgent(
        id=2, power_cap=100, temp_min=-25, temp_max=8, base_cost=12,
        backup_power=True, location_type="Cold",
    ),
    LocationAgent(
        id=3, power_cap=85, temp_min=-20, temp_max=10, base_cost=10,
        backup_power=False, location_type="Cool",
    ),
    LocationAgent(
        id=4, power_cap=90, temp_min=-15, temp_max=12, base_cost=11,
        backup_power=True, location_type="General",
    ),
]

assignment_costs = {
    (1, 1): 18, (1, 2): 20, (1, 3): 25, (1, 4): 22,
    (2, 1): 22, (2, 2): 12, (2, 3): 18, (2, 4): 16,
    (3, 1): 20, (3, 2): 30, (3, 3): 28, (3, 4): 32,
    (4, 1): 15, (4, 2): 10, (4, 3): 14, (4, 4): 11,
    (5, 1): 25, (5, 2): 16, (5, 3): 20, (5, 4): 18,
    (6, 1): 16, (6, 2): 13, (6, 3): 15, (6, 4): 12,
    (7, 1): 30, (7, 2): 35, (7, 3): 40, (7, 4): 38,
    (8, 1): 20, (8, 2): 18, (8, 3): 22, (8, 4): 19,
}

print(f"Autonomous ecosystem example: {len(containers)} containers and {len(locations)} locations")
print("\nContainer agents:")
for container in containers:
    print(
        f"  C{container.id}: {container.cargo_type}, "
        f"cargo value ${container.cargo_value:.0f}K, "
        f"urgency {container.urgency:.1f}, flexibility {container.flexibility:.1f}"
    )

print("\nLocation agents:")
for location in locations:
    backup_status = "Yes" if location.backup_power else "No"
    print(
        f"  L{location.id}: {location.location_type}, "
        f"{location.power_cap} kW, backup power: {backup_status}"
    )

print(f"\nTotal cargo value: ${sum(c.cargo_value for c in containers):.0f}K")

In [ ]:
class AutonomousEcosystem:
    """Market-based simulation for reefer-container assignment."""

    def __init__(
        self,
        containers: List[ContainerAgent],
        locations: List[LocationAgent],
        assignment_costs: Dict[Tuple[int, int], float],
    ) -> None:
        self.containers = containers
        self.locations = locations
        self.assignment_costs = assignment_costs

        self.auctioneer = Auctioneer()
        self.marketplace: Dict[str, Any] = {}
        self.message_bus = deque(maxlen=1000)
        self.contract_registry: Dict[str, Contract] = {}

        self.global_performance = 0.0
        self.iteration = 0
        self.performance_history: List[Dict[str, Any]] = []

        self.max_auction_rounds = 3
        self.auction_timeout = 30
        self.learning_enabled = True

        self._initialize_marketplace()

    def _initialize_marketplace(self) -> None:
        """Set initial market records from the supplied agents."""
        self.marketplace = {
            "active_auctions": [],
            "market_prices": {location.id: location.base_cost for location in self.locations},
            "market_demand": {location.id: 1.0 for location in self.locations},
            "market_supply": {location.id: location.power_cap for location in self.locations},
            "transaction_history": [],
        }

    def run_auction_round(self, round_num: int) -> Dict[str, Any]:
        """Collect offers, coordinate matching, and return this round's results."""
        print(f"\nAuction round {round_num}:")
        self.auctioneer.current_auction = {
            "round": round_num,
            "start_time": time.time(),
            "bids": [],
            "winning_bid": None,
            "contracts": [],
            "total_revenue": 0.0,
        }

        self.message_bus.append(
            Message(
                sender="Auctioneer",
                receiver="All",
                message_type="announcement",
                content=f"Auction round {round_num}: offers are invited for available locations.",
            )
        )

        for container_agent in self.containers:
            if not container_agent.assigned:
                bid = self._generate_bid(container_agent, round_num)
                if bid is not None:
                    self.auctioneer.current_auction["bids"].append(bid)
                    self.message_bus.append(
                        Message(
                            sender=f"Container {container_agent.id}",
                            receiver="Auctioneer",
                            message_type="bid",
                            content=bid,
                        )
                    )

        for location_agent in self.locations:
            if location_agent.power_cap - location_agent.power_usage > 0:
                ask = self._generate_ask_price(location_agent, round_num)
                if ask is not None:
                    self.auctioneer.current_auction["bids"].append(ask)
                    self.message_bus.append(
                        Message(
                            sender=f"Location {location_agent.id}",
                            receiver="Auctioneer",
                            message_type="ask",
                            content=ask,
                        )
                    )

        self._process_auction()
        auction = self.auctioneer.current_auction
        result = {
            "round": round_num,
            "total_revenue": auction["total_revenue"],
            "contracts": list(auction["contracts"]),
            "winning_bid": auction["winning_bid"],
            "participation": len(auction["bids"]),
        }
        self.auctioneer.auction_history.append(result)
        return result

    def _generate_bid(
        self,
        container_agent: ContainerAgent,
        round_num: int,
    ) -> Optional[Dict[str, Any]]:
        """Choose the highest-utility feasible location and calculate a bid."""
        best_location: Optional[LocationAgent] = None
        best_utility = -math.inf

        for location_agent in self.locations:
            available_power = location_agent.power_cap - location_agent.power_usage
            temperature_is_feasible = (
                location_agent.temp_min
                <= container_agent.temp_req
                <= location_agent.temp_max
            )
            if available_power < container_agent.energy or not temperature_is_feasible:
                continue

            remaining_capacity_fraction = available_power / location_agent.power_cap
            location_preference = container_agent.preferences.get(
                location_agent.location_type, 0.5
            )
            utility = (
                0.4 * remaining_capacity_fraction
                + 0.3 * 1.0
                + 0.3 * location_preference
            )

            if utility > best_utility:
                best_utility = utility
                best_location = location_agent

        if best_location is None:
            return None

        base_cost = self.assignment_costs.get(
            (container_agent.id, best_location.id), 100.0
        )
        energy_factor = container_agent.energy / 50.0
        urgency_factor = 1.0 + (50.0 * container_agent.urgency) / 100.0
        time_factor = 1.0 + container_agent.dwell_time / 120.0
        flexibility_discount = 1.0 - (1.0 - container_agent.flexibility) * 0.2
        bid_price = (
            base_cost
            * energy_factor
            * urgency_factor
            * time_factor
            * flexibility_discount
        )

        if bid_price > container_agent.budget:
            return None

        return {
            "container_id": container_agent.id,
            "location_id": best_location.id,
            "price": bid_price,
            "utility": best_utility,
            "bid_type": "bid",
            "round": round_num,
        }

    def _generate_ask_price(
        self,
        location_agent: LocationAgent,
        round_num: int,
    ) -> Optional[Dict[str, Any]]:
        """Calculate a location's ask using its market and capacity state."""
        capacity_utilization = location_agent.power_usage / location_agent.power_cap
        market_demand = self.marketplace["market_demand"].get(location_agent.id, 1.0)
        market_factor = market_demand * location_agent.reputation
        capacity_premium = 2.0 * (1.0 + capacity_utilization)
        ask_price = location_agent.base_cost * market_factor * capacity_premium

        return {
            "location_id": location_agent.id,
            "price": ask_price,
            "utility": 1.0 / ask_price if ask_price > 0 else 0.0,
            "ask_type": "ask",
            "round": round_num,
        }

    def _process_auction(self) -> None:
        """Match compatible bids and asks and register the resulting contracts."""
        auction = self.auctioneer.current_auction
        if auction is None:
            raise RuntimeError("An auction round must be initialized before processing.")

        container_bids = [
            offer for offer in auction["bids"] if offer.get("bid_type") == "bid"
        ]
        location_asks = [
            offer for offer in auction["bids"] if offer.get("ask_type") == "ask"
        ]
        matched_pairs = []

        # Each offer is removed after award, so a container and a location ask
        # can participate in at most one contract in this round.
        while container_bids and location_asks:
            feasible_pairs = []
            for bid in container_bids:
                for ask in location_asks:
                    if bid["location_id"] != ask["location_id"]:
                        continue
                    if bid["price"] < ask["price"]:
                        continue

                    container_agent = next(
                        c for c in self.containers if c.id == bid["container_id"]
                    )
                    location_agent = next(
                        l for l in self.locations if l.id == ask["location_id"]
                    )
                    remaining_power = location_agent.power_cap - location_agent.power_usage
                    if remaining_power >= container_agent.energy:
                        feasible_pairs.append((bid, ask))

            if not feasible_pairs:
                break

            # Prioritize the highest feasible bid, with the lower ask as a tie-break.
            best_bid, best_ask = max(
                feasible_pairs,
                key=lambda pair: (pair[0]["price"], -pair[1]["price"]),
            )
            matched_pairs.append((best_bid, best_ask))
            container_bids.remove(best_bid)
            location_asks.remove(best_ask)

        for container_bid, location_ask in matched_pairs:
            price = (container_bid["price"] + location_ask["price"]) / 2.0
            container_agent = next(
                c for c in self.containers if c.id == container_bid["container_id"]
            )
            location_agent = next(
                l for l in self.locations if l.id == location_ask["location_id"]
            )

            contract = Contract(
                container_id=container_agent.id,
                location_id=location_agent.id,
                price=price,
                duration=container_agent.dwell_time,
            )
            auction["contracts"].append(contract)

            container_agent.assigned = True
            container_agent.current_location = location_agent.id
            container_agent.utility = container_bid["utility"]
            container_agent.satisfaction += 0.1 * (container_agent.utility - 0.5)
            container_agent.experience.append(
                {"round": auction["round"], "location_id": location_agent.id, "price": price}
            )

            location_agent.assigned_containers.append(container_agent.id)
            location_agent.power_usage += container_agent.energy
            location_agent.utilization = location_agent.power_usage / location_agent.power_cap
            location_agent.revenue += price

            contract_key = f"{container_agent.id}_{location_agent.id}"
            self.contract_registry[contract_key] = contract
            self.marketplace["transaction_history"].append(
                {"round": auction["round"], "contract_key": contract_key, "price": price}
            )

        auction["total_revenue"] = sum(
            contract.price for contract in auction["contracts"]
        )
        auction["winning_bid"] = matched_pairs[0][0] if matched_pairs else None

        self.iteration += 1
        if auction["contracts"]:
            average_round_price = auction["total_revenue"] / len(auction["contracts"])
            self.global_performance = 100.0 / (average_round_price / 20.0)

        # Store a snapshot so later plots describe the state at each round rather
        # than repeatedly reading the final, cumulative state.
        self.performance_history.append(
            {
                "iteration": self.iteration,
                "round": auction["round"],
                "total_revenue": auction["total_revenue"],
                "contracts_created": len(auction["contracts"]),
                "global_performance": self.global_performance,
                "average_satisfaction": float(
                    np.mean([c.satisfaction for c in self.containers])
                ),
                "location_states": [
                    {
                        "id": location.id,
                        "utilization": location.utilization,
                    }
                    for location in self.locations
                ],
                "marketplace_state": {
                    "market_prices": dict(self.marketplace["market_prices"]),
                    "total_revenue": sum(
                        contract.price for contract in self.contract_registry.values()
                    ),
                },
            }
        )

    def analyze_emergent_behavior(self) -> Dict[str, Any]:
        """Summarize observed assignment groupings and contract prices."""
        coalition_patterns: Dict[str, List[int]] = defaultdict(list)
        pricing_patterns: Dict[str, List[float]] = defaultdict(list)

        for contract in self.contract_registry.values():
            container_agent = next(
                c for c in self.containers if c.id == contract.container_id
            )
            location_agent = next(
                l for l in self.locations if l.id == contract.location_id
            )

            if container_agent.cargo_type in {"Pharmaceutical", "Vaccines"}:
                group_name = "Pharmaceutical Alliance"
            elif container_agent.cargo_type in {
                "Fresh Produce", "Dairy Products", "Beverages"
            }:
                group_name = "Fresh Produce Cooperative"
            else:
                group_name = "Standard Freight Collective"
            coalition_patterns[group_name].append(location_agent.id)

            if location_agent.location_type in {"Ultra-Cold", "Cold"}:
                pricing_group = "Premium Pricing"
            else:
                pricing_group = "Standard Pricing"
            pricing_patterns[pricing_group].append(contract.price)

        total_revenue = sum(
            contract.price for contract in self.contract_registry.values()
        )
        contract_count = len(self.contract_registry)
        average_price = total_revenue / contract_count if contract_count else 0.0

        return {
            "coalition_patterns": dict(coalition_patterns),
            "pricing_patterns": dict(pricing_patterns),
            "total_revenue": total_revenue,
            "avg_price": average_price,
            "contracts_created": contract_count,
        }

    def get_system_state(self) -> Dict[str, Any]:
        """Return a readable snapshot of agents and marketplace records."""
        container_states = [
            {
                "id": container.id,
                "assigned": container.assigned,
                "location": container.current_location,
                "utility": container.utility,
                "satisfaction": container.satisfaction,
                "urgency": container.urgency,
                "flexibility": container.flexibility,
                "budget": container.budget,
            }
            for container in self.containers
        ]
        location_states = [
            {
                "id": location.id,
                "assigned_containers": list(location.assigned_containers),
                "power_usage": location.power_usage,
                "utilization": location.utilization,
                "revenue": location.revenue,
                "market_price": location.market_price,
                "reputation": location.reputation,
                "location_type": location.location_type,
            }
            for location in self.locations
        ]
        current_auction = self.auctioneer.current_auction
        marketplace_state = {
            "active_auctions": len(current_auction["bids"]) if current_auction else 0,
            "total_contracts": len(self.contract_registry),
            "market_prices": dict(self.marketplace["market_prices"]),
            "total_revenue": sum(
                contract.price for contract in self.contract_registry.values()
            ),
        }
        return {
            "container_states": container_states,
            "location_states": location_states,
            "marketplace_state": marketplace_state,
            "global_performance": self.global_performance,
            "iteration": self.iteration,
            "performance_trend": list(self.performance_history[-10:]),
        }

print("Autonomous ecosystem class is defined.")

In [ ]:
ecosystem = AutonomousEcosystem(containers, locations, assignment_costs)

print("Autonomous ecosystem initialized:")
print(f"  Container agents: {len(ecosystem.containers)}")
print(f"  Location agents: {len(ecosystem.locations)}")
print(f"  Total cargo value: ${sum(c.cargo_value for c in ecosystem.containers):.0f}K")

print("\nRunning auction rounds:")
auction_results = []
for round_num in range(1, 6):
    if all(container.assigned for container in ecosystem.containers):
        break
    result = ecosystem.run_auction_round(round_num)
    auction_results.append(result)
    print(f"  Contracts awarded: {len(result['contracts'])}")
    print(f"  Contract revenue in this round: ${result['total_revenue']:.2f}")
    print(f"  Offers submitted: {result['participation']}")

final_state = ecosystem.get_system_state()
assigned_count = sum(
    state["assigned"] for state in final_state["container_states"]
)
mean_location_utilization = np.mean(
    [state["utilization"] for state in final_state["location_states"]]
)
print("\nCurrent system state:")
print(f"  Assigned containers: {assigned_count}/{len(containers)}")
print(f"  Mean location utilization: {mean_location_utilization:.1%}")
print(f"  Recorded auction rounds: {final_state['iteration']}")
print(
    "  Cumulative contract revenue: "
    f"${final_state['marketplace_state']['total_revenue']:.2f}"
)

# Check that the recorded assignments satisfy the model's basic feasibility rules.
for container in containers:
    if container.assigned:
        location = next(
            location for location in locations
            if location.id == container.current_location
        )
        assert location.temp_min <= container.temp_req <= location.temp_max
        assert container.id in location.assigned_containers

for location in locations:
    assert location.power_usage <= location.power_cap + 1e-9
    assert math.isclose(
        location.utilization,
        location.power_usage / location.power_cap,
        rel_tol=1e-9,
        abs_tol=1e-9,
    )

print("Assignment and capacity checks passed.")

# Plot recorded round-level snapshots; no additional auction rounds are run here.
round_history = ecosystem.performance_history
if round_history:
    rounds = [entry["round"] for entry in round_history]
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("Market-Based Reefer Assignment: Recorded Simulation State", fontsize=15)

    axes[0, 0].plot(
        rounds,
        [entry["total_revenue"] for entry in round_history],
        marker="o",
        color="tab:blue",
    )
    axes[0, 0].set_title("Contract Revenue by Round")
    axes[0, 0].set_xlabel("Auction round")
    axes[0, 0].set_ylabel("Revenue (model price units)")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        rounds,
        [entry["average_satisfaction"] for entry in round_history],
        marker="o",
        color="tab:green",
    )
    axes[0, 1].set_title("Mean Container Satisfaction")
    axes[0, 1].set_xlabel("Auction round")
    axes[0, 1].set_ylabel("Satisfaction (model score)")
    axes[0, 1].grid(True, alpha=0.3)

    utilization_matrix = np.array(
        [
            [state["utilization"] * 100 for state in entry["location_states"]]
            for entry in round_history
        ]
    ).T
    location_labels = [
        f"L{location.id}: {location.location_type}" for location in locations
    ]
    sns.heatmap(
        utilization_matrix,
        annot=True,
        fmt=".1f",
        cmap="RdYlBu_r",
        xticklabels=[f"Round {round_num}" for round_num in rounds],
        yticklabels=location_labels,
        ax=axes[0, 2],
        cbar_kws={"label": "Power-capacity utilization (%)"},
    )
    axes[0, 2].set_title("Location Utilization by Round")
    axes[0, 2].set_xlabel("Auction round")
    axes[0, 2].set_ylabel("Storage location")

    average_market_prices = [
        np.mean(list(entry["marketplace_state"]["market_prices"].values()))
        for entry in round_history
    ]
    axes[1, 0].plot(rounds, average_market_prices, marker="o", color="tab:red")
    axes[1, 0].set_title("Mean Listed Market Price")
    axes[1, 0].set_xlabel("Auction round")
    axes[1, 0].set_ylabel("Price (model units)")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].bar(
        rounds,
        [entry["contracts_created"] for entry in round_history],
        color="skyblue",
    )
    axes[1, 1].set_title("Contracts Awarded by Round")
    axes[1, 1].set_xlabel("Auction round")
    axes[1, 1].set_ylabel("Number of contracts")
    axes[1, 1].grid(True, axis="y", alpha=0.3)

    axes[1, 2].plot(
        rounds,
        [entry["global_performance"] for entry in round_history],
        marker="o",
        color="tab:purple",
    )
    axes[1, 2].set_title("Illustrative Performance Score")
    axes[1, 2].set_xlabel("Auction round")
    axes[1, 2].set_ylabel("Score (model scale)")
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No auction rounds were recorded, so there is no round-level plot to display.")

In [ ]:
coalition_analysis = ecosystem.analyze_emergent_behavior()

print("\nAssignment-pattern and price summary")
print("=" * 52)

print("\nCargo-group assignment patterns:")
if coalition_analysis["coalition_patterns"]:
    for group_name, assigned_location_ids in coalition_analysis["coalition_patterns"].items():
        location_counts = {
            location_id: assigned_location_ids.count(location_id)
            for location_id in sorted(set(assigned_location_ids))
        }
        print(f"  {group_name}: {len(assigned_location_ids)} contract(s)")
        print(f"    Location counts: {location_counts}")
else:
    print("  No contracts were awarded, so no assignment groups are present.")

print("\nContract prices by location category:")
if coalition_analysis["pricing_patterns"]:
    for pricing_group, prices in coalition_analysis["pricing_patterns"].items():
        print(
            f"  {pricing_group}: mean ${np.mean(prices):.2f}, "
            f"standard deviation ${np.std(prices):.2f}"
        )
else:
    print("  No contract prices are available.")

container_count = len(ecosystem.containers)
contract_count = coalition_analysis["contracts_created"]
success_rate = 100.0 * contract_count / container_count if container_count else 0.0
print("\nDescriptive system measures:")
print(f"  Cumulative contract revenue: ${coalition_analysis['total_revenue']:.2f}")
print(f"  Mean contract price: ${coalition_analysis['avg_price']:.2f}")
print(f"  Contracts created: {contract_count}")
print(f"  Assignment rate: {success_rate:.1f}% of containers")

container_satisfactions = [container.satisfaction for container in ecosystem.containers]
location_utilizations = [location.utilization for location in ecosystem.locations]
print("\nAgent-state summaries:")
print(
    f"  Container satisfaction: mean {np.mean(container_satisfactions):.3f}, "
    f"standard deviation {np.std(container_satisfactions):.3f}"
)
print(
    f"  Location utilization: mean {np.mean(location_utilizations):.1%}, "
    f"standard deviation {np.std(location_utilizations):.1%}"
)

total_capacity = sum(location.power_cap for location in ecosystem.locations)
used_capacity = sum(location.power_usage for location in ecosystem.locations)
market_efficiency = 100.0 * used_capacity / total_capacity if total_capacity else 0.0
print(f"  Aggregate power-capacity utilization: {market_efficiency:.1f}%")

# A graph edge represents an awarded container-to-location contract.
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Assignment and Contract Descriptions", fontsize=15)

network_axis = axes[0, 0]
graph = nx.Graph()
for container in ecosystem.containers:
    if container.assigned:
        graph.add_node(f"C{container.id}", node_type="container")
for location in ecosystem.locations:
    if location.assigned_containers:
        graph.add_node(f"L{location.id}", node_type="location")
for contract in ecosystem.contract_registry.values():
    graph.add_edge(
        f"C{contract.container_id}",
        f"L{contract.location_id}",
        weight=contract.price,
    )

if graph.number_of_nodes():
    positions = nx.spring_layout(graph, seed=42)
    container_nodes = [
        node for node, attributes in graph.nodes(data=True)
        if attributes["node_type"] == "container"
    ]
    location_nodes = [
        node for node, attributes in graph.nodes(data=True)
        if attributes["node_type"] == "location"
    ]
    nx.draw_networkx_nodes(
        graph, positions, nodelist=container_nodes,
        node_color="lightblue", node_size=500, ax=network_axis,
        label="Container",
    )
    nx.draw_networkx_nodes(
        graph, positions, nodelist=location_nodes,
        node_color="lightgreen", node_size=600, ax=network_axis,
        label="Location",
    )
    nx.draw_networkx_edges(graph, positions, edge_color="gray", width=1.5, ax=network_axis)
    nx.draw_networkx_labels(graph, positions, ax=network_axis, font_size=9)
    network_axis.legend(loc="best")
else:
    network_axis.text(
        0.5, 0.5, "No contracts were awarded.",
        ha="center", va="center", transform=network_axis.transAxes,
    )
network_axis.set_title("Awarded Container–Location Contracts")
network_axis.axis("off")

price_axis = axes[0, 1]
contract_prices = [
    contract.price for contract in ecosystem.contract_registry.values()
]
if contract_prices:
    price_axis.hist(
        contract_prices, bins=min(10, len(contract_prices)),
        color="skyblue", edgecolor="black", alpha=0.8,
    )
    mean_price = float(np.mean(contract_prices))
    price_axis.axvline(
        mean_price, color="red", linestyle="--",
        label=f"Mean: ${mean_price:.2f}",
    )
    price_axis.legend()
else:
    price_axis.text(
        0.5, 0.5, "No contract prices are available.",
        ha="center", va="center", transform=price_axis.transAxes,
    )
price_axis.set_title("Awarded Contract Prices")
price_axis.set_xlabel("Price (model units)")
price_axis.set_ylabel("Number of contracts")
price_axis.grid(True, alpha=0.3)

satisfaction_axis = axes[1, 0]
satisfaction_axis.hist(
    container_satisfactions, bins=min(8, len(container_satisfactions)),
    color="lightcoral", edgecolor="black", alpha=0.8,
)
satisfaction_axis.set_title("Container Satisfaction Scores")
satisfaction_axis.set_xlabel("Satisfaction (model score)")
satisfaction_axis.set_ylabel("Number of containers")
satisfaction_axis.grid(True, alpha=0.3)

learning_axis = axes[1, 1]
if ecosystem.performance_history:
    recorded_rounds = [entry["round"] for entry in ecosystem.performance_history]
    scores = [entry["global_performance"] for entry in ecosystem.performance_history]
    cumulative_revenues = [
        entry["marketplace_state"]["total_revenue"]
        for entry in ecosystem.performance_history
    ]
    revenue_axis = learning_axis.twinx()
    score_line = learning_axis.plot(
        recorded_rounds, scores, color="tab:blue", marker="o",
        label="Illustrative performance score",
    )
    revenue_line = revenue_axis.plot(
        recorded_rounds, cumulative_revenues, color="tab:red", marker="s",
        label="Cumulative contract revenue",
    )
    learning_axis.set_xlabel("Auction round")
    learning_axis.set_ylabel("Performance score (model scale)", color="tab:blue")
    revenue_axis.set_ylabel("Revenue (model price units)", color="tab:red")
    learning_axis.legend(
        score_line + revenue_line,
        [line.get_label() for line in score_line + revenue_line],
        loc="best",
    )
    learning_axis.grid(True, alpha=0.3)
else:
    learning_axis.text(
        0.5, 0.5, "No auction rounds were recorded.",
        ha="center", va="center", transform=learning_axis.transAxes,
    )
learning_axis.set_title("Recorded Round-Level Measures")

plt.tight_layout()
plt.show()

print("\nInterpretation: these summaries describe the simulation's recorded assignments and prices.")
print("They do not establish learning, equilibrium, global optimality, or comparative performance.")

## Strengths, limitations, and appropriate use

### Strengths of the demonstrated approach

- **Local agent rules:** Containers and locations calculate offers from their own attributes and current state.
- **Negotiated allocation:** A contract is created only when a compatible container bid meets the corresponding location ask.
- **Explicit state changes:** Awarded contracts update assignment status, power use, utilization, revenue, and recorded experience.
- **Inspectable process:** The auction offers, contracts, and round-level snapshots can be examined directly.

### Limitations of this implementation

- **Central coordination:** A single Python object collects offers and selects contracts; this is not a fault-tolerant distributed deployment.
- **Greedy matching:** The auction prioritizes the highest feasible bid and does not guarantee a globally optimal assignment.
- **Simplified market rules:** Bid and ask formulas are illustrative, and the market prices and demand factors are not dynamically learned.
- **Limited learning:** Satisfaction and experience are recorded, but agents do not adapt their strategies from those records.
- **Simplified operations:** The example omits travel, handling, plug availability, electrical-network interactions, uncertain arrivals, and time-dependent refrigeration requirements.
- **No benchmark claim:** The notebook does not compare this simulation with a centralized optimization model, so it cannot establish a performance gap.

### When this modeling perspective may be useful

A market-based agent simulation can help explain how local preferences, budgets, capacity, and negotiation rules influence assignments. It may be worth exploring when decision authority is distributed across stakeholders or locations. Whether it is appropriate for an operational system depends on validated data, safety and service constraints, communication and failure behavior, and comparison against the objectives of the application.

### Further study

Natural extensions within this modeling perspective include testing additional containers and locations, representing time-varying capacity, and examining how alternative bid and ask parameters affect the contracts produced. Such experiments should keep feasibility checks explicit and report the rules and assumptions used. The present notebook provides no numerical result for those extensions.